In [3]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

current_dir = os.getcwd()
sys.path.append(current_dir)
if 'Tuan04' not in current_dir:
    sys.path.append(os.path.join(current_dir, 'Tuan04'))

from src.evaluate import evaluate_model, RESULT_COLUMNS
from src.models import make_models, K_FEATURES, N_SPLITS, RANDOM_STATE

print("Đang nạp dữ liệu...")
X_train = pd.read_csv('data/processed/X_train.csv')
X_test = pd.read_csv('data/processed/X_test.csv')
y_train = pd.read_csv('data/processed/y_train.csv').squeeze('columns')
y_test = pd.read_csv('data/processed/y_test.csv').squeeze('columns')

# Lấy các model đã được tune
models = make_models()

def get_embedded_lasso(model):
    selector = SelectFromModel(
        LassoCV(cv=5, random_state=RANDOM_STATE, n_jobs=-1),
        max_features=K_FEATURES,
        threshold=-np.inf
    )
    return Pipeline([('feature_selection', selector), ('model', model)])

def get_embedded_rf(model):
    rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
    selector = SelectFromModel(rf, threshold=-np.inf, max_features=K_FEATURES)
    return Pipeline([('feature_selection', selector), ('model', model)])

results = []
fitted_pipelines_lasso = {}
fitted_pipelines_rf = {}

print("Đang chạy Embedded Lasso...")
for name, model in models.items():
    pipe_lasso = get_embedded_lasso(model)
    res, fitted_model = evaluate_model(
        'embedded_lasso', name, pipe_lasso, X_train, y_train, X_test, y_test,
        n_splits=N_SPLITS, seed=RANDOM_STATE, return_model=True, detailed=True
    )
    results.append(res)
    fitted_pipelines_lasso[name] = fitted_model 

print("Đang chạy Embedded Random Forest...")
for name, model in models.items():
    pipe_rf = get_embedded_rf(model)
    res, fitted_model = evaluate_model(
        'embedded_rf', name, pipe_rf, X_train, y_train, X_test, y_test,
        n_splits=N_SPLITS, seed=RANDOM_STATE, return_model=True, detailed=True
    )
    results.append(res)
    fitted_pipelines_rf[name] = fitted_model

df_ext = pd.DataFrame(results)
df_ext.to_csv('data/processed/fs_results_embedded_extended.csv', index=False)
df_embedded = df_ext[RESULT_COLUMNS]          # file chuẩn phải đúng 11 cột
df_embedded.to_csv('data/processed/fs_results_embedded.csv', index=False)

print("Đang vẽ biểu đồ so sánh tổng hợp...")
df_baseline = pd.read_csv('data/processed/baseline_results.csv')
df_fw = pd.read_csv('data/processed/fs_results_filter_wrapper.csv')
df_all = pd.concat([df_baseline, df_fw, df_embedded], ignore_index=True)

# Hình 1: RMSE
plt.figure(figsize=(12, 6))
sns.barplot(data=df_all, x='technique', y='cv_log_rmse', hue='model')
plt.title('CV Log RMSE Comparison (All Techniques)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('figures/rmse_by_technique.png', dpi=150)
plt.close()

# Hình 2: Delta Heatmap
pivot_all = df_all.pivot(index='technique', columns='model', values='cv_log_rmse')
baseline_series = pivot_all.loc['baseline']
delta_heatmap = (pivot_all - baseline_series) / baseline_series * 100

plt.figure(figsize=(10, 6))
sns.heatmap(delta_heatmap, annot=True, cmap='RdYlGn_r', center=0, fmt='.2f')
plt.title('Percentage Change in RMSE vs Baseline (%)')
plt.tight_layout()
plt.savefig('figures/delta_heatmap.png', dpi=150)
plt.close()

# Hình 3: Time
plt.figure(figsize=(12, 6))
sns.barplot(data=df_all, x='technique', y='train_seconds', hue='model')
plt.title('Training Time Comparison (seconds)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('figures/time_comparison.png', dpi=150)
plt.close()

# ===== Đồng thuận feature giữa 5 kỹ thuật =====
fw = pd.read_csv('data/processed/selected_features_filter_wrapper.csv', index_col=0)  # index = tên feature

M = pd.DataFrame({
    'filter_kbest': fw['filter_kbest|XGBoost'],      # Filter chọn giống nhau ở mọi mô hình
    'filter_mi': fw['filter_mi|XGBoost'],
    'wrapper_rfe|SVR': fw['wrapper_rfe|SVR'],
    'wrapper_rfe|RandomForest': fw['wrapper_rfe|RandomForest'],
    'wrapper_rfe|XGBoost': fw['wrapper_rfe|XGBoost'],
    'embedded_lasso': fitted_pipelines_lasso['XGBoost'].named_steps['feature_selection'].get_support().astype(int),
    'embedded_rf': fitted_pipelines_rf['XGBoost'].named_steps['feature_selection'].get_support().astype(int),
}, index=X_train.columns)
M.to_csv('data/processed/selected_features_matrix.csv')

# Điểm đồng thuận 0-5 (RFE tính theo tỉ lệ 3 mô hình)
rfe_cols = [c for c in M.columns if c.startswith('wrapper_rfe')]
score = M['filter_kbest'] + M['filter_mi'] + M['embedded_lasso'] + M['embedded_rf'] + M[rfe_cols].mean(axis=1)
df_features = (score.rename('Count').rename_axis('Feature')
                    .reset_index().sort_values('Count', ascending=False))
df_features.to_csv('data/processed/selected_features.csv', index=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_features.head(15), x='Count', y='Feature', color='steelblue')
plt.xlabel('Số kỹ thuật chọn (tối đa 5)')
plt.title('Top Features Agreement Across All 5 Techniques')
plt.tight_layout()
plt.savefig('figures/feature_agreement.png', dpi=150)
plt.close()

print("HOÀN TẤT ĐẾM FEATURE VÀ XUẤT HÌNH!")

Đang nạp dữ liệu...
Đang chạy Embedded Lasso...
Đang chạy Embedded Random Forest...
Đang vẽ biểu đồ so sánh tổng hợp...
HOÀN TẤT ĐẾM FEATURE VÀ XUẤT HÌNH!
